# 08 - Scaling: Prefill Conditions

Run the full 2x2 matrix (self_prefill, cross_prefill, paraphrase_self, paraphrase_cross)
for each Qwen3 model size. Generates inline paraphrases for non-4B models.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import gc, torch
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from lib.data import load_gsm8k
from lib.prefill import run_prefill_condition
from lib.paraphrase import paraphrase_cots

dataset = load_gsm8k()

def model_tag(name):
    return name.split("/")[-1].lower().replace("-", "_")

def load_cots_for_model(model_name):
    tag = model_tag(model_name)
    cot_dir = COT_CACHE if model_name == PRIMARY_MODEL else CACHE_DIR / f"cots_{tag}"
    cots = {}
    for p in cot_dir.glob("*.json"):
        r = json.loads(p.read_text())
        cots[r["problem_id"]] = r["cot_text"]
    return cots

In [ ]:
for model_name in SCALING_MODELS:
    tag = model_tag(model_name)
    print(f"\n{'='*60}")
    print(f"Model: {model_name} ({tag})")
    print(f"{'='*60}")

    cot_lookup = load_cots_for_model(model_name)
    print(f"COTs available: {len(cot_lookup)}")
    if not cot_lookup:
        print("No COTs found, skipping.")
        continue

    # --- Self-prefill with this model ---
    print(f"\nLoading {model_name} for self-prefill...")
    try:
        llm = LLM(model=model_name, dtype="bfloat16", max_model_len=4096)
    except Exception as e:
        print(f"Failed to load: {e}")
        continue

    run_prefill_condition(llm, f"self_prefill_{tag}", model_name, dataset, cot_lookup)

    # --- Paraphrase self: need paraphrases for this model's COTs ---
    # For 4B, reuse existing paraphrase_light. For others, generate inline.
    para_condition = f"paraphrase_light_{tag}"
    if model_name == PRIMARY_MODEL:
        # Reuse existing
        para_lookup = {}
        for p in PARAPHRASE_CACHE.glob("paraphrase_light_*.json"):
            r = json.loads(p.read_text())
            para_lookup[r["problem_id"]] = r["paraphrased_cot"]
    else:
        # Check if already generated
        para_lookup = {}
        for p in PARAPHRASE_CACHE.glob(f"{para_condition}_*.json"):
            r = json.loads(p.read_text())
            para_lookup[r["problem_id"]] = r["paraphrased_cot"]

        if len(para_lookup) < len(cot_lookup):
            print(f"Generating paraphrases for {tag} COTs...")
            del llm; gc.collect(); torch.cuda.empty_cache()

            para_llm = LLM(model=PARAPHRASER_MODEL, dtype="bfloat16", max_model_len=4096)
            para_tok = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
            cot_list = [{"problem_id": pid, "cot_text": txt} for pid, txt in cot_lookup.items()]
            paraphrase_cots(para_llm, para_tok, cot_list, para_condition, heavy=False)
            del para_llm, para_tok; gc.collect(); torch.cuda.empty_cache()

            # Reload
            para_lookup = {}
            for p in PARAPHRASE_CACHE.glob(f"{para_condition}_*.json"):
                r = json.loads(p.read_text())
                para_lookup[r["problem_id"]] = r["paraphrased_cot"]

            # Reload self model
            llm = LLM(model=model_name, dtype="bfloat16", max_model_len=4096)

    run_prefill_condition(llm, f"paraphrase_self_{tag}", model_name, dataset, para_lookup)

    del llm; gc.collect(); torch.cuda.empty_cache()

    # --- Cross-prefill and paraphrase-cross with Gemma ---
    print(f"\nLoading {CROSS_MODEL} for cross-prefill...")
    llm = LLM(model=CROSS_MODEL, dtype="bfloat16", max_model_len=4096)

    run_prefill_condition(llm, f"cross_prefill_{tag}", CROSS_MODEL, dataset, cot_lookup)
    run_prefill_condition(llm, f"paraphrase_cross_{tag}", CROSS_MODEL, dataset, para_lookup)

    del llm; gc.collect(); torch.cuda.empty_cache()
    print(f"{model_name} complete.")

In [ ]:
# Summary
for model_name in SCALING_MODELS:
    tag = model_tag(model_name)
    conditions = [f"self_prefill_{tag}", f"cross_prefill_{tag}",
                  f"paraphrase_self_{tag}", f"paraphrase_cross_{tag}"]
    for cond in conditions:
        n = len(list(PREFILL_CACHE.glob(f"{cond}_*.json")))
        print(f"{cond:45s}: {n}")